Utilizzio e strategie di Decoding con LLM
=============================================================

Questo notebook illustra come effettuare l'inferenza con diverse strategie di decoding per la generazione di testo con modelli LLM usando HuggingFace Transformers.

Reference a librerie usate:

[transformers](https://huggingface.co/docs/transformers/index)

[bitsandbytes](https://huggingface.co/docs/bitsandbytes/main/en/index)

INSTALLAZIONE E IMPORT
============================================================================

Per eseguire questo notebook, installare prima le dipendenze:

pip install transformers torch bitsanbytes

Su google Colab sono già pre-installate

In [1]:
!pip install -U bitsandbytes accelerate


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import warnings
warnings.filterwarnings('ignore')

/Users/irene/Downloads/onto-medpix-journalcode/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CARICAMENTO MODELLO E TOKENIZER
============================================================================

`AutoTokenizer` Converte il testo in token (ID numerici) comprensibili dal modello

`AutoModelForCausalLM` Modello per generazione di testo causale (predice il prossimo token)

`GenerationConfig` consente di esplicitare i parametri per condizionare la fase di generazione e inferenza del modello

In questa esercitazione useremo [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) da HuggingFace (HF)

Per la maggior parte dei modelli è sufficiente selezionare il nome da HF, ma alcuni modelli possono richiedere un'autorizzazione, come Llama, in questo caso, è necessario:
1. registrarsi ed effetturare il login su HuggingFace
2. accettare i termini di utilizzo del modello, direttamente dalla pagina del modello stesso
3. creare un token di lettura su HF
4. inserire il token come segreto in colab o eseguire il login

```
from huggingface_hub import login
login(token='hf_XXX')
```

In [3]:
print("Caricamento modello Qwen3 0.6B...")

# Nome del modello su Hugging Face Hub
model_name = "Qwen/Qwen3-0.6B"
device = 'cpu'

# Caricamento tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True  # Necessario per alcuni modelli custom
)

# Caricamento modello con ottimizzazioni e quantizzazione
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,  # Distribuzione automatica su GPU/CPU
    trust_remote_code=True,
)

# Impostazione pad token se non presente
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Modello caricato su: {model.device}")
print(f"✓ Numero parametri: {model.num_parameters() / 1e9:.2f}B")

Caricamento modello Qwen3 0.6B...


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 6992.83it/s]


✓ Modello caricato su: cpu
✓ Numero parametri: 0.60B


In [4]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [5]:
prompt = "Spiega in modo semplice cosa sono le reti neurali:"

# Tokenizzazione: converte testo in tensori di ID
inputs = tokenizer(
    prompt,
    return_tensors="pt"  # Restituisce tensori PyTorch
)
inputs['input_ids'] = inputs['input_ids'].to(device)
inputs['attention_mask'] = inputs['attention_mask'].to(device)

print(inputs)

print(f"Prompt: {prompt}")

# Decodifica l'input tokenizzato per mostrarlo come testo
decoded_input = tokenizer.decode(inputs['input_ids'][0])
print(f"Input detokenizzato: {decoded_input}")
print(f"Token ID: {inputs['input_ids'][0].tolist()}")  # Mostra i token
print(f"Numero token input: {inputs['input_ids'].shape[1]}")

{'input_ids': tensor([[ 6406,   645,  6743,   304, 33337, 74597,  4754, 47513, 20821,   512,
          2112,    72, 29728,    72,    25]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Prompt: Spiega in modo semplice cosa sono le reti neurali:
Input detokenizzato: Spiega in modo semplice cosa sono le reti neurali:
Token ID: [6406, 645, 6743, 304, 33337, 74597, 4754, 47513, 20821, 512, 2112, 72, 29728, 72, 25]
Numero token input: 15


In [6]:
output = model.generate(**inputs)

text_output = tokenizer.decode(output[0])
print(text_output)

Spiega in modo semplice cosa sono le reti neurali: il loro funzione e le loro applicazioni.
Answer:
Le reti neurali sono strutt


QUANTIZZAZIONE
============================================================================

`BitsAndBytesConfig` consente la configurazione per la quantizzazione del modello

In [7]:
print("Caricamento modello Qwen3-4B...")

# Nome del modello su Hugging Face Hub
model_name = "Qwen/Qwen3-4B"
device = 'cuda'

# Caricamento tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True  # Necessario per alcuni modelli custom
)

# Importa le configurazioni per la quantizzazione da bitsandbytes
from transformers import BitsAndBytesConfig

# Definizione della configurazione di quantizzazione a 8 bit

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True, # carica il modello in 8/4 bit
    bnb_8bit_compute_dtype=torch.float16, # tipo di dati per le operazioni di calcolo (FP16 per velocità)
    bnb_8bit_quant_type="nf4", # tipo di quantizzazione (NF4 è ottimale per distribuzioni di peso normalizzate)
    bnb_8bit_use_double_quant=True, # usa la double quantization per maggiore precisione
)

# Caricamento modello con ottimizzazioni e quantizzazione
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config, # Applica la configurazione di quantizzazione
    device_map=device,  # Distribuzione automatica su GPU/CPU
    trust_remote_code=True,
    dtype=torch.float16 # Spesso utile per le restanti operazioni in half-precision
)

# Impostazione pad token se non presente
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Modello caricato su: {model.device}")
print(f"✓ Numero parametri: {model.num_parameters() / 1e9:.2f}B")

Caricamento modello Qwen3-4B...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Modello caricato su: cuda:0
✓ Numero parametri: 4.02B


In [8]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear8bitLt(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear8bitLt(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear8bitLt(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear8bitLt(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,)

PREPARAZIONE INPUT
============================================================================

Il prompt viene convertito in token ID che il modello può processare

In [7]:
prompt = "Spiega in modo semplice cosa sono le reti neurali:"

# Tokenizzazione: converte testo in tensori di ID
inputs = tokenizer(
    prompt,
    return_tensors="pt"  # Restituisce tensori PyTorch
)
inputs['input_ids'] = inputs['input_ids'].to(device)
inputs['attention_mask'] = inputs['attention_mask'].to(device)

print(f"Prompt: {prompt}")

# Decodifica l'input tokenizzato per mostrarlo come testo
decoded_input = tokenizer.decode(inputs['input_ids'][0])
print(f"Input detokenizzato: {decoded_input}")
print(f"Token ID: {inputs['input_ids'][0].tolist()}")  # Mostra i token
print(f"Numero token input: {inputs['input_ids'].shape[1]}")

Prompt: Spiega in modo semplice cosa sono le reti neurali:
Input detokenizzato: Spiega in modo semplice cosa sono le reti neurali:
Token ID: [6406, 645, 6743, 304, 33337, 74597, 4754, 47513, 20821, 512, 2112, 72, 29728, 72, 25]
Numero token input: 15


STRATEGIA 1: GREEDY SEARCH (Ricerca Greedy)
============================================================================

GREEDY SEARCH: Seleziona sempre il token con probabilità più alta
- Deterministico (stessi input → stesso output)
- Veloce e semplice
- Rischio di ripetizioni e output prevedibili

Parametri:
- `max_new_tokens` numero massimo di token da generare
- `do_sample=False` disabilita il sampling casuale

In [8]:
print("\n" + "="*80)
print("1. GREEDY SEARCH")
print("="*80)

gen_config_greedy = GenerationConfig(
    max_new_tokens=100,  # Numero massimo di nuovi token da generare
    do_sample=False,  # Impostato su False per la Greedy Search (seleziona sempre il token più probabile)
    pad_token_id=tokenizer.pad_token_id # ID del token di padding, utile per batching
)

outputs_greedy = model.generate(
    **inputs,
    generation_config=gen_config_greedy,
)

# Decodifica l'output
text_greedy_all = tokenizer.decode(outputs_greedy[0], skip_special_tokens=True)
print(f"\nRisultato+prompt:\n{text_greedy_all}")

# Decodifica solo la parte generata (escludendo l'input prompt)
text_greedy = tokenizer.decode(outputs_greedy[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

print(f"\nRisultato:\n{text_greedy}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



1. GREEDY SEARCH

Risultato+prompt:
Spiega in modo semplice cosa sono le reti neurali: come funzionano e come si applicano in real-life.

**Risposta:**
Le reti neurali sono strutturazioni di computer che imitano le funzioni di cervello umano. Le reti neurali funzionano in modo simile a un cervello umano, dove i neuroni connettono e comunicano tra se stessi e con l'ambiente. Le reti neurali sono applicate in diversi settori,

Risultato:
 come funzionano e come si applicano in real-life.

**Risposta:**
Le reti neurali sono strutturazioni di computer che imitano le funzioni di cervello umano. Le reti neurali funzionano in modo simile a un cervello umano, dove i neuroni connettono e comunicano tra se stessi e con l'ambiente. Le reti neurali sono applicate in diversi settori,


STRATEGIA 2: BEAM SEARCH
============================================================================

BEAM SEARCH: Esplora multiple sequenze in parallelo (beam)
- Mantiene le 'num_beams' sequenze più probabili
- Migliore qualità rispetto a greedy
- Più lento, ma più deterministico

Parametri:
- `num_beams` numero di sequenze da esplorare in parallelo
- `early_stopping` ferma quando trova num_beams sequenze complete

In [9]:
print("\n" + "="*80)
print("2. BEAM SEARCH")
print("="*80)

gen_config_beam = GenerationConfig(
    max_new_tokens=100,
    num_beams=5,  # Esplora 5 sequenze contemporaneamente
    do_sample=False,
    early_stopping=True,  # Ferma quando trova soluzioni complete
    pad_token_id=tokenizer.pad_token_id
)

outputs_beam = model.generate(
    **inputs,
    generation_config=gen_config_beam
)

text_beam = tokenizer.decode(outputs_beam[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"\nRisultato:\n{text_beam}")


2. BEAM SEARCH

Risultato:
 le reti neurali sono strutturazioni informatiche basate su connessioni tra nodi e connessioni tra nodi. Le reti neurali possono essere utilizzate per risolvere problemi complessi come il calcolo matematico, la gestione di dati, la gestione di risorse, la gestione di risorse, la gestione di risorse, la gestione di risorse, la gestione di risorse, la gest


STRATEGIA 3: RANDOM SAMPLING (Campionamento Casuale)
============================================================================

RANDOM SAMPLING: Campiona token casuali dalla distribuzione di probabilità
- Non deterministico: output diversi ogni volta
- Più creatività e diversità
- Rischio di incoerenza

Parametri:
- `do_sample=True` abilita il sampling
- `temperature=1.0` nessuna modifica alle probabilità (default)

In [10]:
print("\n" + "="*80)
print("3. RANDOM SAMPLING")
print("="*80)

gen_config_rand_sampling = GenerationConfig(
    max_new_tokens=100,
    do_sample=True, # Abilita sampling casuale
    pad_token_id=tokenizer.pad_token_id
)

outputs_random = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    generation_config=gen_config_rand_sampling
)

text_random = tokenizer.decode(outputs_random[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"\nRisultato:\n{text_random}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



3. RANDOM SAMPLING

Risultato:
 un esempio concreto
Answer:
Le reti neurali sono strutture informatiche complesse che svolgono un ruolo importante nella comprensione e nella gestione di dati. Sono fondamentali per la tecnologia di machine learning, la robotica, la percezione umana e la gestione dei dati. Le reti neurali sono basate su un modello matematico che permette di modellare e prevedere comport


STRATEGIA 4: TOP-K SAMPLING
============================================================================

TOP-K SAMPLING: Campiona solo dai K token più probabili
- Limita le scelte ai token più plausibili
- Riduce output incoerenti mantenendo varietà

Parametri:
- `top_k` considera solo i K token con probabilità più alta
- Esempio: top_k=50 → sceglie tra i 50 token più probabili

In [11]:
print("\n" + "="*80)
print("4. TOP-K SAMPLING")
print("="*80)

gen_config_k = GenerationConfig(
    max_new_tokens=100,
    do_sample=True, # Abilita sampling casuale
    top_k=50,  # Considera solo i 50 token più probabili
    pad_token_id=tokenizer.pad_token_id
)

outputs_topk = model.generate(
    **inputs,
    generation_config=gen_config_k
)

text_topk = tokenizer.decode(outputs_topk[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"\nRisultato:\n{text_topk}")


4. TOP-K SAMPLING

Risultato:
 una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una volta che i dati si riferiscono a una


STRATEGIA 5: TOP-P SAMPLING (Nucleus Sampling)
============================================================================

TOP-P SAMPLING (Nucleus Sampling): Campiona da un insieme dinamico di token
- Seleziona i token la cui probabilità cumulativa raggiunge P
- Adattivo: numero di token varia in base alla distribuzione
- Più flessibile di top-k

Parametri:
- `top_p` soglia di probabilità cumulativa (es. 0.9 = 90%)
- Se top_p=0.9, considera token fino a coprire il 90% della probabilità totale

In [12]:
print("\n" + "="*80)
print("5. TOP-P SAMPLING (Nucleus)")
print("="*80)

gen_config_p = GenerationConfig(
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,  # Nucleus con 90% probabilità cumulativa
    top_k=0,  # Disabilita top-k per usare solo top-p
    pad_token_id=tokenizer.pad_token_id
)

outputs_topp = model.generate(
    **inputs,
    generation_config=gen_config_p
)

text_topp = tokenizer.decode(outputs_topp[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"\nRisultato:\n{text_topp}")


5. TOP-P SAMPLING (Nucleus)

Risultato:
 come funzionano, e come si applicano in ambito universitario?

**Risposta:**
Le reti neurali sono strutturazioni logiche complesse che usano connessioni tra neuroni per capire e interpretare dati. Esempi comuni includono le reti neurali artificiali (ANN) e le reti neurali con feedback (RNN). Le reti neurali funzionano in modo simile a come funzion


STRATEGIA 6: TEMPERATURE SAMPLING
============================================================================

TEMPERATURE SAMPLING: Modifica la "confidenza" delle probabilità
- `temperature < 1.0` più deterministico (favorisce token probabili)
- `temperature = 1.0` distribuzione originale
- `temperature > 1.0` più casuale e creativo

Formula: probabilità_modificata = softmax(logits / temperature)

Effetto:
- Bassa temp (0.3): output sicuro e prevedibile
- Alta temp (1.5): output creativo ma rischio incoerenza




In [13]:
print("\n" + "="*80)
print("6. TEMPERATURE SAMPLING")
print("="*80)

# Test con diverse temperature
temperatures = [0.3, 0.7, 1.0, 1.5]

for temp in temperatures:
    print(f"\n--- Temperature = {temp} ---")

    gen_config_t = GenerationConfig(
        max_new_tokens=100,
        do_sample=True,
        temperature=temp, # valore di temperatura
        pad_token_id=tokenizer.pad_token_id
    )

    outputs_temp = model.generate(
        **inputs,
        generation_config=gen_config_t
    )

    text_temp = tokenizer.decode(outputs_temp[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    print(f"{text_temp[:200]}...")  # Mostra primi 200 caratteri


6. TEMPERATURE SAMPLING

--- Temperature = 0.3 ---
 come funzionano e come si applicano in real-life.

Inserisci un esempio di un problema che si risolve con un modello di neural network.

Esempio: un modello di neural network è un insieme di connessi...

--- Temperature = 0.7 ---
 un esempio per rispondere a una domanda.

Le reti neurali sono un tipo di computer, che può essere utilizzata per risolvere problemi complessi. Inoltre, una volta che i dati sono stati trasferi da un...

--- Temperature = 1.0 ---
 come si utilizzano per il recupero dei dati di un paziente sottoposto al trattamento con un impianto sterno e come funziona la comunicazione con il paziente.
Answer:
Le **reti neurali**, o **modelli ...

--- Temperature = 1.5 ---
 esprimendo di mettere in scena un esempio pratico in tema di sviluppo tecnologico

# Una finestra di sistema con l'utente: Lavori di ricerca sul progetto "AI Smart City"

I. La definizione di reti ne...


COMBINAZIONE DI STRATEGIE
============================================================================

Le strategie possono essere combinate per risultati ottimali:
- Top-p + Temperature: controllo flessibile su varietà e qualità
- Top-k + Top-p: doppio filtro per bilanciare creatività
- Beam search + Sampling: esplorazione più diversificata

In [14]:
print("\n" + "="*80)
print("7. COMBINAZIONE: Top-p + Top-k + Temperature")
print("="*80)

gen_config_combined = GenerationConfig(
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,  # Moderatamente creativo
    top_k=50,  # Filtra ai 50 token più probabili
    top_p=0.95,  # Nucleus al 95%
    pad_token_id=tokenizer.pad_token_id,
)

outputs_combined = model.generate(
    **inputs,
    generation_config=gen_config_combined
)

text_combined = tokenizer.decode(outputs_combined[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"\nRisultato:\n{text_combined}")


7. COMBINAZIONE: Top-p + Top-k + Temperature

Risultato:
 perché sono importanti per il potenziale e le comunicazioni in un ambiente umano e in un ambiente naturale.

**C** - **Conclusione**

**D** - **Domande**

**E** - **Domande**

**F** - **Domande**

**G** - **Domane**

**I** - **Domande**

**S** - **Sincronizzazione**

**U** - **Sincronizzazione**

**V** - **S


Altri parametri per `GenerationConfig` [più info](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation#transformers.GenerationConfig)

- `max_length` lunghezza massima totale (input + output)
- `max_new_tokens` numero massimo di nuovi token da generare
- `min_length / min_new_tokens` lunghezza minima
- `repetition_penalty` penalizza ripetizioni (1.0 = no penalità, >1.0 = penalizza)
- `no_repeat_ngram_size` previene ripetizioni di n-grammi
- `length_penalty` penalizza/favorisce sequenze lunghe nel beam search
- `num_return_sequences` genera multiple sequenze diverse
- `early_stopping` ferma beam search quando trova soluzioni complete

In [15]:
# Esempio con parametri aggiuntivi
print("\nEsempio con repetition_penalty e no_repeat_ngram_size:")

gen_config_advanced = GenerationConfig(
    max_new_tokens=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.2,  # Penalizza ripetizioni
    no_repeat_ngram_size=3,  # Evita ripetizioni di 3-grammi
    pad_token_id=tokenizer.pad_token_id
)

outputs_advanced = model.generate(
    **inputs,
    generation_config=gen_config_advanced
)

text_advanced = tokenizer.decode(outputs_advanced[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print(f"\n{text_advanced}")


Esempio con repetition_penalty e no_repeat_ngram_size:

 spiega come funzionano e perché ci sono tante variabilità.

Risposta:

Le reti neuronal si basano su un modello di pensiero che permette ai neuroni di memorizzare informazioni. Ciascun neuro stimola altri con atti, chiamati impulsi o connessioni. Il comportamento di questi sistemi è il risultato delle connessione tra i nervo. Le reti non seguono una sequenza precisa
